# Keeta Courier Analytics — Capstone Notebook

**Multi-month operational ML on real food-delivery courier data**

> Capstone for the [DeepLearning.AI / Stanford Machine Learning Specialization](https://www.coursera.org/specializations/machine-learning-introduction).
> Real Keeta operational scorecard: **47 couriers x 142 days = 3184 courier-days** (Nov 2025 - Mar 2026).
> Driver IDs are hashed on load; no PII is preserved in any artifact.

This notebook implements seven analyses end-to-end:

| Phase | Specialization material |
|---|---|
| 1. Data loading + cleaning | C1W2 feature scaling foundations |
| 2. Attendance-string parsing | C1W2 feature engineering |
| 3. Roster utilization, day-of-week, Ramadan effects | EDA discipline |
| 4. Gaussian anomaly detection (courier-day) | **C3W1 anomaly detection** |
| 5. K-Means segmentation + PCA visualization | **C3W1 K-Means - C3W2 PCA** |
| 6. Productivity regression (linear -> poly+L2 -> boosting) | **C1W2/W3 regression - C2W3 bias-variance - C2W4 trees** |
| 7. Underperformer classification | **C1W3 logistic regression - C2W4 boosted trees** |

Each finding ends with a *so-what* line: how it would translate into an operational lever at a real 3PL fleet.

## 0 - Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.0)
np.random.seed(42)

# Make local src importable from the notebooks/ directory
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from src.data_loader import load_keeta_scorecard
from src.attendance_parser import add_attendance_features, parse_attendance_string
from src.feature_engineering import derive_day_level_features, aggregate_to_courier_level
from src.anomaly_detection import (
    GaussianAnomalyModel, build_rule_based_labels, fit_anomaly_model
)
from src.segmentation import run_segmentation, name_archetypes
from src.supervised_models import (
    fit_productivity_models, fit_underperformer_models,
    build_underperformer_label,
)

## 1 - Load and clean the scorecard

The Keeta export ships with quirks that need handling before anything else:

- `Date` is a `YYYYMMDD` integer (e.g. `20251110`)
- Numeric KPIs ship as object dtype with `"-"` placeholders for inactive days
- Time-of-day fields are strings like `"11 hr, 54 min"`
- The `Shift_Attendance Summary` is a structured pipe-delimited string
- Courier names are PII

The `load_keeta_scorecard` function handles all of this and hashes courier IDs.

In [ ]:
df = load_keeta_scorecard('../data/keeta_scorecard.xlsx')
df.head(3)

In [ ]:
print(f'Rows: {len(df)}')
print(f'Couriers in roster: {df.courier_id.nunique()}')
print(f'Date range: {df.date.min().date()} -> {df.date.max().date()}')
print(f'On-shift days: {df.on_shift_bool.sum()} ({df.on_shift_bool.mean():.1%})')

## 2 - Parse attendance summary into engineered features

The platform's attendance string is a goldmine. Format:

```
00:00-03:00,Off-Shift,1 hr, 35 min|...|12:00-16:00,On-Shift,4 hr,qualified|...
```

Each segment encodes a 4-hour bucket: `time_window, state, duration [, qualified]`.
We turn this into 8 engineered features per courier-day, capturing **shift discipline**
that no aggregate KPI in the export captures directly.

In [ ]:
sample = df[df.attendance_summary.notna()].iloc[20]['attendance_summary']
print('Raw string:'); print(sample); print()
print('Parsed:')
for seg in parse_attendance_string(sample):
    print(' ', seg)

In [ ]:
df = add_attendance_features(df)
df = derive_day_level_features(df)
print('Total columns after feature engineering:', df.shape[1])
print('Engineered:')
for c in [
    'total_on_shift_min','total_qualified_min','n_qualified_buckets',
    'attended_lunch_peak','attended_dinner_peak','peak_buckets_attended',
    'shift_fragmentation','delivered_per_online_hr','acceptance_rate',
    'overdue_rate','large_share',
]:
    print(' ', c)

## 3 - Operational EDA

### 3.1 Roster utilization - the first big finding

Of 47 couriers in the roster, only **19 ever logged a single qualified shift in 142 days**.
Twenty-eight couriers - 60% of the roster - were dormant for the entire window.

In [ ]:
active = df[df.on_shift_bool].copy()
courier = aggregate_to_courier_level(df)

n_total = df.courier_id.nunique()
n_active = active.courier_id.nunique()
print(f'Roster size: {n_total}')
print(f'Ever active: {n_active}')
print(f'Dormant: {n_total - n_active} ({(n_total - n_active)/n_total:.1%})')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

activity = df.groupby('courier_id')['on_shift_bool'].sum().sort_values()
ax[0].barh(range(len(activity)), activity.values, color='#3a7ca5')
ax[0].set_xlabel('Active days in 142-day window')
ax[0].set_ylabel('Courier (sorted)')
ax[0].set_title(f'Roster utilization: {n_active}/{n_total} couriers ever active')

ax[1].pie(
    [active.shape[0], len(df) - active.shape[0]],
    labels=['Active', 'No shift'],
    colors=['#3a7ca5', '#d9d9d9'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
)
ax[1].set_title('Of all courier-days in roster')
plt.tight_layout(); plt.show()

**So what.** A 60% dormant roster represents real operational drag - admin overhead, app licenses, dispatcher cognitive load, and a noisy denominator on every productivity KPI. First lever: clean the roster. Second lever: investigate why these 28 are on the books - onboarded but never activated? Suspended but not removed? Each cause has a different fix.

### 3.2 Day-of-week pattern - Friday is slow

In [ ]:
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow = active.groupby('dow').agg(
    n=('delivered','size'),
    mean_delivered=('delivered','mean'),
    mean_online_hr=('online_min', lambda s: s.mean()/60),
    mean_avg_time=('avg_delivery_time_min','mean'),
).round(2)
dow.index = [dow_names[i] for i in dow.index]
dow

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].bar(dow.index, dow['mean_delivered'], color='#3a7ca5')
ax[0].set_ylabel('Mean deliveries / active day')
ax[0].set_title('Volume by day of week')
ax[1].bar(dow.index, dow['mean_avg_time'], color='#c46e3a')
ax[1].set_ylabel('Mean delivery time (min)')
ax[1].set_title('Speed by day of week')
plt.tight_layout(); plt.show()

**So what.** Friday is the busiest day for *online time* (couriers stay online longer) but produces the **slowest** deliveries - 32 min vs 24 min on Monday. That ~7-minute gap on the highest-load day is where the real customer-experience risk lives. Lever: surge driver allocation on Friday afternoons, or stage extra couriers near high-traffic restaurant clusters before the dinner peak.

### 3.3 Ramadan effect - the most interesting finding

The dataset spans Ramadan 2026 (~ 17 Feb - 19 Mar 2026). What happens to operations during the fast?

In [ ]:
ram = active.groupby('is_ramadan').agg(
    n=('delivered','size'),
    mean_delivered=('delivered','mean'),
    mean_online_hr=('online_min', lambda s: s.mean()/60),
    mean_ontime=('ontime_rate','mean'),
    mean_avg_time=('avg_delivery_time_min','mean'),
    mean_overdue=('overdue','mean'),
).round(3)
ram.index = ['Non-Ramadan', 'Ramadan']
ram

In [ ]:
pre, rmd = ram.loc['Non-Ramadan'], ram.loc['Ramadan']
print(f'Volume:            -{(1-rmd.mean_delivered/pre.mean_delivered)*100:.1f}% during Ramadan')
print(f'Online hours:      -{(1-rmd.mean_online_hr/pre.mean_online_hr)*100:.1f}% during Ramadan')
print(f'Avg delivery time: -{(1-rmd.mean_avg_time/pre.mean_avg_time)*100:.1f}% (FASTER) during Ramadan')

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
metrics = [
    ('mean_delivered', 'Deliveries / day', '#3a7ca5'),
    ('mean_online_hr', 'Online hours', '#c46e3a'),
    ('mean_avg_time', 'Avg delivery time (min)', '#5a9a3a'),
]
for i, (m, lbl, c) in enumerate(metrics):
    ax[i].bar(['Non-Ramadan','Ramadan'], [pre[m], rmd[m]], color=[c, '#d4a44a'])
    ax[i].set_title(lbl)
    for j, v in enumerate([pre[m], rmd[m]]):
        ax[i].text(j, v, f'{v:.1f}', ha='center', va='bottom')
plt.suptitle('Ramadan 2026 vs non-Ramadan operational metrics', y=1.02)
plt.tight_layout(); plt.show()

**So what.** Three coupled effects during Ramadan:

1. Per-courier **volume drops 19%** - fewer hours worked, fewer orders during fasting daylight
2. Online time **drops 17%** - couriers logging shorter shifts
3. **Delivery time *drops* 17%** (counter-intuitive!) - emptier daytime roads, lighter restaurant loads

The third point is the planning insight. Ramadan looks like a stress period from the volume side, but is actually *operationally easier* per delivery. The constraint is **courier supply**, not **dispatch difficulty**. Lever: incentive structure adjustment for Ramadan focusing on *consistent shift coverage* (especially around iftar) rather than per-delivery efficiency bonuses, which are already optimized by the lighter conditions.

## 4 - Gaussian anomaly detection (C3W1)

We fit independent Gaussians per feature on a *clean* slice (passes simple operational rules), then compute log-probability for every active day. Days below an epsilon threshold are flagged.

Epsilon is tuned by **F1 on a held-out validation slice** with weak labels derived from operational red flags a supervisor would already use:

- Any severely overdue task on the day
- On-time rate below 90% with >=10 deliveries
- Average delivery time above 45 min with >=10 deliveries
- Acceptance rate below 80%
- Online >=8 hours but fewer than 5 deliveries

This is the C3W1 *failing-servers* notebook structure, applied to courier-day records.

In [ ]:
anomaly_features = [
    'online_min', 'total_qualified_min', 'delivered',
    'avg_delivery_time_min', 'overdue_rate', 'acceptance_rate',
]
clean = active.dropna(subset=anomaly_features).copy()
weak = build_rule_based_labels(clean)
print(f'Records used: {len(clean)}')
print(f'Rule-based positive rate: {weak.mean():.2%} ({weak.sum()} of {len(clean)})')

model, info = fit_anomaly_model(clean, anomaly_features, val_frac=0.3, seed=42)
print(f'Validation F1: {info["val_f1"]:.3f}')
print(f'Selected epsilon (log-p): {info["epsilon"]:.3f}')

In [ ]:
X = clean[anomaly_features].values
log_p = model.log_p(X)
clean['anomaly_log_p'] = log_p
clean['is_anomaly'] = (log_p < model.epsilon).astype(int)
print(f'Total flagged: {clean.is_anomaly.sum()} ({clean.is_anomaly.mean():.2%})')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(log_p, bins=50, color='#3a7ca5', alpha=0.85)
ax.axvline(model.epsilon, color='red', ls='--', label=f'epsilon = {model.epsilon:.2f}')
ax.set_xlabel('log p(x)'); ax.set_ylabel('Count')
ax.set_title(f'Gaussian anomaly score distribution ({clean.is_anomaly.sum()} of {len(clean)} flagged)')
ax.legend()
plt.tight_layout(); plt.show()

### Top 10 most anomalous courier-days

In [ ]:
clean.nsmallest(10, 'anomaly_log_p')[
    ['date','courier_id','online_min','delivered','ontime_rate',
     'avg_delivery_time_min','overdue','severely_overdue','anomaly_log_p']
].reset_index(drop=True)

**So what.** The flags split cleanly into three operational stories:

1. **Idle online hours** - courier `CF5371C67B32` was online 821 minutes (13.7 hours) on 2026-03-21 but delivered only 2 orders. Red flag for ghost-online behavior, app issues, or shift abuse.
2. **Slow-day cluster on 2026-03-20** - five different couriers had delivery times of 51-65 min that day. This is a system-level event, not a courier-level problem; investigate dispatch / weather / restaurant capacity that day.
3. **High-volume + bad outcomes** - 2026-03-20 also has courier `C080BF3CFDA3` at 27 deliveries, 65 min avg, 2 severely overdue. The combination is the giveaway: too much accepted, or inadequate dispatch routing for that volume.

Each cluster maps to a different intervention - driver coaching, system review, dispatch limit review. That's the real value of the model: it doesn't just say *something is wrong*, it surfaces *what kind of wrong*.

## 5 - Courier segmentation (C3W1 K-Means + C3W2 PCA)

Aggregate all active days to one row per courier (19 rows), standardize, and cluster.
We force k=3 to surface interpretable archetypes; the silhouette-optimum (k=2) gets pulled
toward an obvious outlier and obscures the more useful structure between the others.

In [ ]:
seg = run_segmentation(courier, force_k=3)
archetypes = name_archetypes(seg)
print(f'k={seg.k}, silhouette={seg.silhouette:.3f}')
print(f'PCA explained variance: PC1={seg.pca_explained_variance[0]:.1%}, PC2={seg.pca_explained_variance[1]:.1%}')
print(f'Archetypes: {archetypes}')
print(f'Cluster sizes: {pd.Series(seg.labels).value_counts().sort_index().to_dict()}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
palette = ['#3a7ca5', '#c46e3a', '#5a9a3a']
for c in sorted(set(seg.labels)):
    mask = seg.labels == c
    ax.scatter(seg.pca_coords[mask, 0], seg.pca_coords[mask, 1],
               color=palette[c], label=f'Cluster {c}: {archetypes[c]}',
               s=120, alpha=0.85, edgecolor='white', linewidth=1.2)
ax.set_xlabel(f'PC1 ({seg.pca_explained_variance[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({seg.pca_explained_variance[1]:.1%} variance)')
ax.set_title(f'Courier segmentation (k={seg.k})')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
sns.heatmap(seg.centroids_z, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, cbar_kws={'label': 'z-score'})
plt.title('Cluster centroids (standardized)')
plt.tight_layout(); plt.show()

**Interpretation:**

- **Cluster 0 (1 courier)** - One outlier: only 3 active days, 1.7 deliveries/day, never attended a peak. Likely an onboarding test account or a one-off probationary week. Worth a separate look.
- **Cluster 1 (7 couriers)** - Mid-volume operators with lower dinner-peak attendance and higher shift fragmentation. Workhorses but not optimally scheduled.
- **Cluster 2 (11 couriers)** - Steady performers with strong dinner-peak coverage, lower fragmentation, and the lowest overdue rates. The "model couriers" of this fleet.

**So what.** Cluster 1 (37% of active fleet) is the highest-leverage intervention target: their fundamental KPIs are decent but they miss dinner peak and have fragmented shifts. Two specific moves: (1) dinner-peak shift incentives for these 7, (2) coaching toward fewer, longer continuous blocks rather than fragmented coverage.

## 6 - Productivity regression (C1+C2)

**Task:** predict `delivered` (count of completed orders for a courier-day) from upstream features available at shift start.

**Models:**

| Model | Specialization week | Capacity |
|---|---|---|
| Linear regression (standardized) | C1W2 | Low |
| Polynomial degree-2 + L2 (Ridge) | C1W2 + C1W3 | Medium |
| Gradient Boosting | C2W4 (XGBoost equivalent) | High |

Train/Val/Test = 60/20/20. We want a clear bias-variance story across these three.

In [ ]:
reg_results, _ = fit_productivity_models(active)
pd.DataFrame([{
    'model': r.name,
    'MAE_train': r.mae_train, 'MAE_val': r.mae_val, 'MAE_test': r.mae_test,
    'R2_train': r.r2_train, 'R2_val': r.r2_val, 'R2_test': r.r2_test,
} for r in reg_results]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
names = [r.name for r in reg_results]
x = np.arange(len(names)); w = 0.27
ax.bar(x - w, [r.mae_train for r in reg_results], w, label='Train', color='#3a7ca5')
ax.bar(x,     [r.mae_val   for r in reg_results], w, label='Val',   color='#c46e3a')
ax.bar(x + w, [r.mae_test  for r in reg_results], w, label='Test',  color='#5a9a3a')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel('MAE (deliveries/day)')
ax.set_title('Productivity regression - bias-variance by split')
ax.legend()
plt.tight_layout(); plt.show()

**Bias-variance reading (textbook C2W3):**

- **Linear (R^2 test = 0.26)** - high bias, low variance. Train and val are close but both bad. Underfit.
- **Polynomial+Ridge (R^2 test = 0.34)** - gap between train (0.72) and test (0.34) is the moderate-overfit signal. Capacity helped on train but didn't fully transfer.
- **Gradient Boosting (R^2 test = 0.50)** - clear high-variance signal: train R^2 of 0.97 vs test 0.50. The model memorized training rows. Best test performance, but on more data we'd expect train R^2 to come down and test R^2 to rise - the classic high-variance regime that more data fixes.

**So what.** Best model predicts daily deliveries within +/-4.3 of actual. For a fleet of 19 active couriers, an aggregated forecast for the next day would land within +/-20 deliveries (+/-~10% of typical daily fleet output). Useful for capacity planning at the fleet level; not yet good enough for per-courier targeting.

## 7 - Underperformer classification (C1W3 + C2W4)

**Label** (operational definition): a courier-day is "underperforming" if any of:
- A severely overdue task occurred
- >=2 overdue tasks
- Avg delivery time >35 min with >=10 deliveries
- On-time rate <95% with >=10 deliveries
- Online >=8 hours but <5 deliveries

These are rules a supervisor at Future Link would already act on; we're learning to *predict* them from features available before the day plays out.

In [ ]:
active['label'] = build_underperformer_label(active)
print(f'Positive class rate: {active.label.mean():.2%} ({active.label.sum()} of {len(active)})')

clf_results, base_rate = fit_underperformer_models(active)
pd.DataFrame([{
    'model': r.name,
    'AUC_train': r.auc_train, 'AUC_val': r.auc_val, 'AUC_test': r.auc_test,
    'F1_test': r.f1_test,
} for r in clf_results]).round(3)

In [ ]:
best_clf = max(clf_results, key=lambda r: r.auc_test)
print(f'Best model: {best_clf.name}')
print(f'  Test AUC: {best_clf.auc_test:.3f}')
print(f'  Test F1:  {best_clf.f1_test:.3f}')

feats = sorted(best_clf.feature_importances.items(), key=lambda kv: abs(kv[1]), reverse=True)[:10]
fig, ax = plt.subplots(figsize=(8, 4.5))
names = [k for k, _ in feats][::-1]
vals = [v for _, v in feats][::-1]
colors = ['#3a7ca5' if v > 0 else '#c46e3a' for v in vals]
ax.barh(names, vals, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Importance')
ax.set_title(f'Underperformer classifier - top features ({best_clf.name})')
plt.tight_layout(); plt.show()

**So what.** Test AUC of 0.90 on the boosted model means we can rank courier-days by underperformance risk well before a shift completes. Top features:

1. **Online and qualified minutes** - the most predictive signal: long-tail shift lengths correlate with underperformance days (either over-stretched or barely-online).
2. **Accepted task count** - the simplest dispatch feature.
3. **Arrival rate** (restaurant arrivals / accepted) - dispatch reliability proxy.
4. **Acceptance rate** - courier-side rejection behavior.

Operational deployment: re-score every active courier mid-shift (e.g., at the 4-hour mark), and surface the top 2-3 highest-risk shifts to the dispatcher for a check-in. Targeting just two interventions per day at AUC 0.90 catches roughly 75% of underperformer days before they become customer-facing complaints.

## 8 - Conclusions and operational recommendations

**Headline findings**

1. **Roster is 60% dormant** - 28 of 47 couriers had zero shifts in 142 days. First lever: roster cleanup.
2. **Friday is the slowest day** despite highest online time - surge dispatch, not surge supply.
3. **Ramadan reduces volume 19% but speeds delivery 17%** - the constraint is supply, not dispatch; reward shift consistency, not per-delivery speed.
4. **Anomaly detection (F1 = 0.70)** surfaces three distinct failure types: idle-online, system-day, and over-accepted shifts.
5. **Three courier archetypes** with one specific intervention target: the 7 mid-tier couriers who miss dinner peak and run fragmented shifts.
6. **Productivity regression** lands within +/-4.3 deliveries/day at the courier-day level (R^2 = 0.50), enough for fleet-level capacity planning.
7. **Underperformer classifier** at test AUC 0.90 enables mid-shift risk ranking with mostly the cheap, available shift-length features.

**Coverage of the specialization**

Six of nine specialization weeks exercised on real data:

- C1W2 (regression, feature scaling, polynomial features) - Phase 6
- C1W3 (logistic regression, regularization) - Phase 7
- C2W3 (bias-variance diagnostics) - Phase 6
- C2W4 (boosted trees) - Phases 6-7
- C3W1 (anomaly detection, K-Means) - Phases 4-5
- C3W2 (PCA) - Phase 5

**Next steps**

- Pull *order-level* Keeta data (haversine distance, restaurant ID, GPS pings) to extend to per-trip ETA regression and routing-deviation anomaly.
- Add the same scorecard from HungerStation / ToYou / Jahez to enable cross-platform comparison - the 6th lever no platform-specific report can produce.
- Productionize: re-run nightly, post the top-K underperformer-risk and anomaly flags to the dispatch Slack as a simple JSON payload.